## Merge data to proper format JSON file for validation

1) Get matched eng - kyr senses
2) Get gold english senses
3) Iter over sentences and form sentence for validation set

In [2]:
import json
import pandas as pd

In [ ]:
with open('../auxiliary_data/apertium_constitution.json') as f:
    kyr_sentences = json.load(f)
    
# file with info of english wsd queries stored: sentence id, kyr word id, eng lemma etc.
with open('../deepseek_answers/deepseek_eng_results.json') as f:
    eng_queries = json.load(f)

eng_gold = pd.read_csv('../gold_data/eng_gold.csv')
matched_senses = pd.read_csv('../gold_data/eng-kyr-matched_gold.csv')
glossary_csv = pd.read_csv('../auxiliary_data/glossary.csv')

In [ ]:
from collections import defaultdict

senses_dct = defaultdict()
# get indexes of kyr senses matched to eng senses
for _, row in matched_senses.iterrows():
    selected_option = int(row.selected_option.split('.')[0])
    # numbering from 1, when right answer is absent -1
    senses_dct[(row.eng_sense, row.kyr_lemma)] = selected_option - 1 if selected_option > 0 else selected_option

In [ ]:
senses_gold = []

# extract keys from eng_gold
for opt in eng_gold.selected_option:
    ans = opt.split(';')[0]
    senses_gold.append(ans)

### Sentences are stored as a result of apertium parsing, because of problems with apertium in MacOS

In [ ]:
from streamparser import LexicalUnit

# get LexicalUnit from string
def get_lex_units(units):
    return [LexicalUnit(unit) for unit in units]

In [10]:
kyr_units = []
for sent in kyr_sentences:
    processed_sent = get_lex_units(sent)
    kyr_units.append(processed_sent)

In [17]:
kyr_units[0][0].readings

[[SReading(baseform='биз', tags=['prn', 'pers', 'p1', 'pl', 'nom'])],
 [SReading(baseform='биз', tags=['prn', 'pers', 'p1', 'pl', 'nom']),
  SReading(baseform='э', tags=['cop', 'aor', 'p3', 'pl'])],
 [SReading(baseform='биз', tags=['prn', 'pers', 'p1', 'pl', 'nom']),
  SReading(baseform='э', tags=['cop', 'aor', 'p3', 'sg'])]]

In [ ]:
from math import isnan
from collections import defaultdict

# get dict {kyr lemma -> list of senses} preserving order from dictionary 
glossary_dict = defaultdict()
for _, row in glossary_csv[1:].iterrows():
    senses = []
    for i in range(1, 12):
        if type(row[f'sense{i}']) == float and isnan(row[f'sense{i}']):
            break
        senses.append(row[f'sense{i}'])
    glossary_dict[row.noun] = senses

In [ ]:
# few cases where apertium finds multiple lemmas, wrong cases are filtered manually
hard_dct = {
    frozenset(('ата', 'атан')): 'ата',
    frozenset(('бал', 'бала')): 'бала',
    frozenset(('жара', 'жаран')): 'жаран',
    frozenset(('күн', 'күнү')): 'күн',
    frozenset(('уй', 'уюм')): 'уюм'
}

In [ ]:
def get_sentence(sentence, kyr_token_id):
    target_unit = sentence[kyr_token_id]
    nouns = set()

    # iterate over apertiums analysis and extract all lemmas
    for reading in target_unit.readings:
        for sread in reading:
            if 'n' in sread.tags:
                # extracting baseform of kyrgyz word
                nouns.add(sread.baseform)

    # if there is only one lemma everything is ok
    if len(nouns) == 1:
        noun = nouns.pop()
    else:
        # few lemmas - one of our hard cases
        noun = hard_dct[frozenset(nouns)]

    # get sentence word by word and add placeholders
    sentence_split = [unit.wordform for unit in sentence]
    sentence_split.insert(kyr_token_id + 1, "[TGT]")
    sentence_split.insert(kyr_token_id, "[TGT]")

    return noun, " ".join(sentence_split)

In [ ]:
validation_set = []
cnt = 0

for query, eng_sense in zip(eng_queries, senses_gold):
    # create instance_id
    sentence_id, kyr_token_id = query['sentence_id'], query['kyr_token_id']
    sentence = kyr_units[sentence_id]

    lemma, tgt_sentence = get_sentence(sentence, kyr_token_id)

    # get senses
    kyr_senses = glossary_dict[lemma]
    sense_idx = senses_dct[(eng_sense, lemma)]

    # if there is only one definition or no right definition -- skip
    if len(kyr_senses) == 1 or sense_idx == -1:
        continue

    for ind, kyr_sense in enumerate(kyr_senses):
        label = int(ind == sense_idx)
        validation_set.append(
            {
                "instance_id": f"{sentence_id}_{kyr_token_id}",
                "context": tgt_sentence,
                "gloss": kyr_sense,
                "label": label
            }
        )
    cnt += 1

In [60]:
validation_set[-1]

{'instance_id': '517_9',
 'context': 'Конституцияны кабыл алуу , Конституцияга өзгөртүүлөрдү жана толуктоолорду киргизүү жөнүндө [TGT] мыйзамга [TGT] Президент тарабынан кол коюлат .',
 'gloss': 'Күз болгондо жел менен кошо учкан ак түстүү желе сыяктуу үлпүлдөк нерсе.',
 'label': 0}

In [61]:
with open('validation_set.json', 'w', encoding="utf-8") as f:
    json.dump(validation_set, f, ensure_ascii=False, indent=4)